# Final Project — Week 1: From Raw Files to a Curated Dataset

Welcome to the first week of this Learning Pod project! ✴️

We will use the public MovieLens dataset to practise PySpark together and learn how to explore and prepare real-world data.

This week, we will explore the raw files, practise preparing data, and build a **curated dataset**. Along the way, we will use analysis questions to learn how the data is structured and what it can tell us.

By the end of this notebook you will have:

- explored the raw files using **RDDs**
- loaded them into **DataFrames** with explicit schemas
- checked the **data quality** and cleaned the data
- **joined** the four datasets together
- explored questions about the dataset with `groupBy`, aggregations and SQL
- written a curated dataset to disk, which you will use for more learning in Week 2

Everything you need was covered in modules **01 - Spark Introduction** and **02 - PySpark DataFrames**. Whenever you get stuck, go back to those notebooks: they contain every function used here.

> Read the project [README](../README.md) before starting.

## 0. Setup

Let's start, as always, by creating a `SparkSession` and importing everything we need.

In [ ]:
import os
import time
import urllib.request
import zipfile

import pyspark.sql.functions as f
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType,
    LongType,
)

spark = SparkSession.builder.appName("movielens-project-week1").getOrCreate()
sc = spark.sparkContext

## 1. Getting the data

We will work with the **MovieLens small** dataset, published by the [GroupLens](https://grouplens.org/datasets/movielens/) research group. It contains 100.836 ratings and 3.683 tags applied to 9.742 movies by 610 users between March 1996 and September 2018.

The dataset is made of four files:

| File | Columns | Description |
| --- | --- | --- |
| `ratings.csv` | `userId`, `movieId`, `rating`, `timestamp` | One row per rating. Ratings go from 0.5 to 5.0 stars |
| `movies.csv` | `movieId`, `title`, `genres` | One row per movie. The title ends with the release year and the genres are pipe separated |
| `tags.csv` | `userId`, `movieId`, `tag`, `timestamp` | Free text tags applied by users to movies |
| `links.csv` | `movieId`, `imdbId`, `tmdbId` | Identifiers of the same movie on IMDb and TMDB |

Run the cell below to download and unzip the dataset. It is only ~1 MB, so it should take a couple of seconds.

If you are behind a firewall and the download fails, download the file manually from [https://grouplens.org/datasets/movielens/](https://grouplens.org/datasets/movielens/) (link *ml-latest-small.zip*) and unzip it into the `04_final_project/movielens-data/` folder.

In [ ]:
DATA_DIR = "../movielens-data"
DATASET_DIR = f"{DATA_DIR}/ml-latest-small"
DATASET_URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

if not os.path.exists(DATASET_DIR):
    os.makedirs(DATA_DIR, exist_ok=True)
    zip_path = f"{DATA_DIR}/ml-latest-small.zip"

    urllib.request.urlretrieve(DATASET_URL, zip_path)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(DATA_DIR)

sorted(os.listdir(DATASET_DIR))

## 2. Warming up with RDDs

Before jumping into DataFrames, let's get a first feeling for the data using the lowest level API Spark offers: **RDDs**.

Reading a file with `sc.textFile()` gives you an RDD where each element is one line of text, with no schema and no parsing whatsoever. That is exactly what we want here: it forces us to think about what the raw data actually looks like.

1. Load `movies.csv` into an RDD, print the first line and count how many lines the file has.

Hint: `sc.textFile()` and the actions `first()` and `count()`.

In [ ]:
movies_rdd = sc.textFile(f"{DATASET_DIR}/movies.csv")

print(movies_rdd.first())
print(movies_rdd.count())

2. The first line of the file is the header, which is not data. Create a new RDD without it.

Hint: store the header in a variable and use `filter()` to remove it. Then confirm the new RDD has exactly one element less.

In [ ]:
header = movies_rdd.first()

movie_lines = movies_rdd.filter(lambda line: line != header)

print(movie_lines.first())
print(movie_lines.count())

3. How many movies are there per genre? Answer using **only RDD transformations and actions**.

Let's break this question down:
- Each line looks like `1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy`
- The genres are in the last field, separated by `|`
- One movie can belong to several genres, so one line must produce **several** pairs `(genre, 1)`
- Those pairs then have to be added up per genre

> ⚠️ Careful: some titles contain commas, like `11,"American President, The (1995)",Comedy|Drama|Romance`. Splitting a line by `,` and taking the field at index 2 would break for those movies. Since the genres are always the **last** field, take the element at index `-1` instead.

Hint: `map()`, `flatMap()` and `reduceByKey()`.

In [ ]:
genre_counts = (
    movie_lines
    .map(lambda line: line.split(",")[-1])
    .flatMap(lambda genres: genres.split("|"))
    .map(lambda genre: (genre, 1))
    .reduceByKey(lambda count_a, count_b: count_a + count_b)
)

print(genre_counts.count())
genre_counts.collect()

4. Which are the 10 most common genres?

Hint: check the [`sortBy()`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.sortBy.html) documentation and the action `take()`.

In [ ]:
genre_counts.sortBy(lambda genre_count: -genre_count[1]).take(10)

5. RDDs are evaluated **lazily**: the whole chain of transformations is recomputed from scratch every time you call an action. On a file this small you would never notice it, so let's pretend that parsing a line is expensive.

Run the cell below and then time two consecutive `count()` actions on `slow_genres`. Next, `cache()` the RDD and time two more.

What changed, and why?

In [ ]:
def slow_split(line):
    """Pretend that parsing a line is expensive."""
    time.sleep(0.0005)
    return line.split(",")[-1]


slow_genres = movie_lines.map(slow_split)

In [ ]:
def time_count(rdd, label):
    start = time.time()
    rdd.count()
    print(f"{label}: {time.time() - start:.2f}s")


time_count(slow_genres, "1st count, no cache ")
time_count(slow_genres, "2nd count, no cache ")

slow_genres.cache()

time_count(slow_genres, "1st count, cached   ")
time_count(slow_genres, "2nd count, cached   ")

The first three counts all pay the full parsing cost: the two uncached ones because nothing is stored, and the first cached one because `cache()` is itself lazy, it only fills the cache while the action runs. From the second cached count on, Spark reads the result straight from memory.

Caching is worth it when an RDD is **reused** and **expensive to compute**. Caching everything just fills up the memory of your executors.

Working like this is painful: we had to parse the file ourselves, worry about the header and be careful with commas inside quotes. That is exactly the problem DataFrames solve, so let's move on to them.

## 3. Loading the data into DataFrames

In modules 01 and 02 we always read CSV files with `inferSchema`. That is convenient, but it has two costs:

- Spark has to **read the whole file once** just to guess the types, before reading it again to load the data
- The guess can be **wrong**, and you only find out much later

In a production pipeline you want to state the schema explicitly. From now on, that is what we will do.

> ⚠️ `imdbId` looks like a number, but it has leading zeros (`0114709`). What happens if you read it as an integer? Think about it before choosing its type.

6. Define a `StructType` schema for each of the four files and load them into the DataFrames `df_ratings`, `df_movies`, `df_tags` and `df_links`.

The `read_csv` helper below is already written for you, note that it uses `.schema()` instead of the `inferSchema` option.

Hint: the timestamps are unix timestamps, i.e. the number of seconds since 1970-01-01, so they do not fit in an `IntegerType`.

In [ ]:
def read_csv(file_name, schema):
    return (
        spark.read.format("csv")
        .option("header", "true")
        .option("sep", ",")
        .schema(schema)
        .load(f"{DATASET_DIR}/{file_name}")
    )

In [ ]:
ratings_schema = StructType(
    [
        StructField("userId", IntegerType(), False),
        StructField("movieId", IntegerType(), False),
        StructField("rating", DoubleType(), False),
        StructField("timestamp", LongType(), False),
    ]
)

movies_schema = StructType(
    [
        StructField("movieId", IntegerType(), False),
        StructField("title", StringType(), True),
        StructField("genres", StringType(), True),
    ]
)

tags_schema = StructType(
    [
        StructField("userId", IntegerType(), False),
        StructField("movieId", IntegerType(), False),
        StructField("tag", StringType(), True),
        StructField("timestamp", LongType(), False),
    ]
)

# imdbId is kept as a string because it has meaningful leading zeros
links_schema = StructType(
    [
        StructField("movieId", IntegerType(), False),
        StructField("imdbId", StringType(), True),
        StructField("tmdbId", IntegerType(), True),
    ]
)

df_ratings = read_csv("ratings.csv", ratings_schema)
df_movies = read_csv("movies.csv", movies_schema)
df_tags = read_csv("tags.csv", tags_schema)
df_links = read_csv("links.csv", links_schema)

df_movies.printSchema()
df_movies.show(5, truncate=False)

7. How many ratings, movies, tags and users are there? Check that the numbers match the ones announced in the dataset documentation.

In [ ]:
n_ratings = df_ratings.count()
n_movies = df_movies.count()
n_tags = df_tags.count()
n_users = df_ratings.select("userId").distinct().count()

print(f"{n_ratings} ratings, {n_movies} movies, {n_tags} tags, {n_users} users")

assert (n_ratings, n_movies, n_tags, n_users) == (100836, 9742, 3683, 610)

## 4. Data quality checks

It is a best practice to check a dataset before trusting it, to avoid bad surprises later. Let's run a few checks that you should run on any new dataset you get.

8. A user should not be able to rate the same movie twice, and a `movieId` should appear only once in `movies.csv`. Check whether that is true.

Hint: compare `count()` with the count after `dropDuplicates()`.

In [ ]:
print("ratings:", df_ratings.count(), "unique:", df_ratings.dropDuplicates(["userId", "movieId"]).count())
print("movies:", df_movies.count(), "unique:", df_movies.select("movieId").distinct().count())

9. How many nulls does each column of `df_links` have?

In [ ]:
df_links.select(
    [f.count(f.when(f.col(column).isNull(), column)).alias(column) for column in df_links.columns]
).show()

10. Ratings are supposed to go from 0.5 to 5.0 stars, in 0.5 increments. Confirm it, and look at how the ratings are distributed.

Hint: `summary()` gives you the descriptive statistics of a column, and a `groupBy()` on the rating itself gives you the distribution.

In [ ]:
df_ratings.select("rating").summary().show()

df_ratings.groupBy("rating").count().orderBy("rating").show()

## 5. Cleaning and enriching

The raw data hides some information inside text and numbers that we cannot use directly:

- the **release year** is glued to the title, as in `Toy Story (1995)`
- the **dates** are stored as unix timestamps, as in `964982703`

Let's extract both. This part introduces three functions that were not used in the previous modules:

| Function | What it does |
| --- | --- |
| [`f.regexp_extract(col, pattern, group)`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.regexp_extract.html) | Returns the group of a regex match, or an empty string if there is no match |
| [`f.regexp_replace(col, pattern, replacement)`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.regexp_replace.html) | Replaces every regex match |
| [`f.from_unixtime(col)`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.from_unixtime.html) | Converts a unix timestamp into a timestamp string |

11. Create `df_movies_clean` with a `release_year` integer column extracted from the title, and with the year removed from the `title` column.

Let's break this question down:
- The year is always the last thing in the title, between parentheses: the regex `\((\d{4})\)$` captures it
- Not every movie has a year: `regexp_extract` returns an empty string in that case, which should become `null`
- Once extracted, the year should be removed from the title, and the leftover whitespace trimmed

How many movies have no release year?

In [ ]:
df_movies_clean = (
    df_movies
    .withColumn("release_year", f.regexp_extract(f.col("title"), r"\((\d{4})\)$", 1))
    .withColumn(
        "release_year",
        f.when(f.col("release_year") == "", None).otherwise(f.col("release_year").cast(IntegerType())),
    )
    .withColumn("title", f.trim(f.regexp_replace(f.col("title"), r"\(\d{4}\)$", "")))
)

df_movies_clean.show(5, truncate=False)

print("movies without release year:", df_movies_clean.filter(f.col("release_year").isNull()).count())
print("movies without genres:", df_movies_clean.filter(f.col("genres") == "(no genres listed)").count())

12. Create `df_ratings_clean` by replacing the `timestamp` column with a `rating_date` date column, plus the `rating_year` and `rating_month` columns.

Then check the first and the last date in the dataset.

Hint: `f.to_date(f.from_unixtime(...))`, and then `f.year()` and `f.month()`, which you already used in module 02.

In [ ]:
df_ratings_clean = (
    df_ratings
    .withColumn("rating_date", f.to_date(f.from_unixtime(f.col("timestamp"))))
    .withColumn("rating_year", f.year(f.col("rating_date")))
    .withColumn("rating_month", f.month(f.col("rating_date")))
    .drop("timestamp")
)

df_ratings_clean.show(5)

df_ratings_clean.select(f.min("rating_date"), f.max("rating_date")).show()

13. Tags are free text typed by users, so `Funny`, `funny` and `funny ` are all different strings even though they mean the same thing.

Create `df_tags_clean` with the tags lowercased and trimmed, a `tag_date` column, and without the rows where the tag is null.

How many distinct tags are there before and after the cleaning?

In [ ]:
print("distinct tags before cleaning:", df_tags.select("tag").distinct().count())

df_tags_clean = (
    df_tags
    .withColumn("tag", f.lower(f.trim(f.col("tag"))))
    .withColumn("tag_date", f.to_date(f.from_unixtime(f.col("timestamp"))))
    .drop("timestamp")
    .dropna(subset=["tag"])
)

print("distinct tags after cleaning:", df_tags_clean.select("tag").distinct().count())

df_tags_clean.show(5)

## 6. Joining everything together

We now have three clean datasets, but they only become useful when put together.

14. Add the IMDb and TMDB identifiers to the movies, creating `df_movies_full`.

Every movie should be kept even if it has no identifiers, and the resulting DataFrame must have exactly as many rows as `df_movies_clean`.

Which join type should you use?

In [ ]:
df_movies_full = df_movies_clean.join(df_links, on=["movieId"], how="left")

print(df_movies_full.count(), "==", df_movies_clean.count())
df_movies_full.show(5, truncate=False)

15. Create `df`, the analysis DataFrame, by joining the ratings with the movies. Every rating must have its movie information.

Cache it, since we are going to use it in every question that follows.

In [ ]:
df = df_ratings_clean.join(df_movies_full, on=["movieId"], how="inner").cache()

df.printSchema()
print(df.show(5, False))
print(df.count())

16. `movies.csv` only contains movies that have at least one rating **or** one tag. How many movies were never rated? And how many were never tagged?

Hint: a `left_anti` join keeps only the rows of the left DataFrame that have **no** match on the right one.

In [ ]:
rated_movie_ids = df_ratings_clean.select("movieId").distinct()
tagged_movie_ids = df_tags_clean.select("movieId").distinct()

print("never rated:", df_movies_clean.join(rated_movie_ids, on=["movieId"], how="left_anti").count())
print("never tagged:", df_movies_clean.join(tagged_movie_ids, on=["movieId"], how="left_anti").count())

## 7. Questions to explore

Time to explore what this dataset can tell us.

17. Which are the 10 most rated movies, and what is their average rating?

In [ ]:
df_movie_stats = (
    df
    .groupBy("movieId", "title")
    .agg(
        f.count("rating").alias("n_ratings"),
        f.round(f.avg("rating"), 3).alias("avg_rating"),
    )
)

df_movie_stats.orderBy(f.desc("n_ratings")).show(10, truncate=False)

18. Which are the 10 best rated movies?

Run it first without any filter and look at the result. Does it look useful? Now repeat it keeping only movies with **at least 50 ratings** and compare.

This is a problem you will meet in every ratings dataset: averages computed over very few observations are not comparable to averages computed over hundreds of them. We will come back to it in Week 2.

In [ ]:
df_movie_stats.orderBy(f.desc("avg_rating")).show(10, truncate=False)

(
    df_movie_stats
    .filter(f.col("n_ratings") >= 50)
    .orderBy(f.desc("avg_rating"))
    .show(10, truncate=False)
)

19. How many ratings were submitted each year? Was the platform more active in the beginning or at the end of the period?

In [ ]:
(
    df
    .groupBy("rating_year")
    .agg(
        f.count("*").alias("n_ratings"),
        f.countDistinct("userId").alias("n_active_users"),
        f.round(f.avg("rating"), 3).alias("avg_rating"),
    )
    .orderBy("rating_year")
    .show(30)
)

20. Who are the 10 most active users? For each of them, show how many ratings they submitted, how many distinct movies they rated and their average rating.

Is the most active user also a generous one?

In [ ]:
(
    df
    .groupBy("userId")
    .agg(
        f.count("rating").alias("n_ratings"),
        f.countDistinct("movieId").alias("n_movies"),
        f.round(f.avg("rating"), 3).alias("avg_rating"),
    )
    .orderBy(f.desc("n_ratings"))
    .show(10)
)

21. Do older movies get better ratings than recent ones? Compute the number of ratings and the average rating per **release decade**.

Hint: the decade of a year can be computed with an integer division, for instance `1995 -> 1990`.

In [ ]:
(
    df
    .filter(f.col("release_year").isNotNull())
    .withColumn("decade", (f.col("release_year") / 10).cast(IntegerType()) * 10)
    .groupBy("decade")
    .agg(
        f.count("rating").alias("n_ratings"),
        f.countDistinct("movieId").alias("n_movies"),
        f.round(f.avg("rating"), 3).alias("avg_rating"),
    )
    .orderBy("decade")
    .show(30)
)

22. Which genres are the most rated, and which ones get the best ratings? Answer this one using **Spark SQL** instead of the DataFrame API.

Let's break this question down:
- A movie can have several genres, all packed in a single string like `Adventure|Animation|Children`
- In SQL, `SPLIT(genres, '\\|')` turns that string into an array and `EXPLODE()` turns each element of the array into its own row
- Only genres with more than 1000 ratings are relevant here

Hint: register `df` as a temporary view with `createOrReplaceTempView()` and then use `spark.sql()`.

In [ ]:
df.createOrReplaceTempView("ratings_movies")

spark.sql(
    r"""
    SELECT
        genre,
        COUNT(*) AS n_ratings,
        ROUND(AVG(rating), 3) AS avg_rating
    FROM (
        SELECT EXPLODE(SPLIT(genres, '\\|')) AS genre, rating
        FROM ratings_movies
    )
    GROUP BY genre
    HAVING COUNT(*) > 1000
    ORDER BY avg_rating DESC
    """
).show()

## 8. The deliverable: a curated dataset

This is the most important part of the week. Everything you cleaned so far lives in memory and will be gone as soon as you shut down the notebook. Week 2 starts from this deliverable, so it has to be written to disk.

We will save it as **Parquet** instead of CSV. Parquet is a columnar format that keeps the schema inside the file, compresses the data, and lets Spark read only the columns a query actually needs. In short: you will never have to declare a schema again when reading it back.

23. Write `df_movies_full`, `df_ratings_clean` and `df_tags_clean` to `../curated/`, each one in its own folder.

Use the `overwrite` mode so you can re-run this cell as many times as you want.

Hint: `df.write.mode("overwrite").parquet(path)`.

In [ ]:
CURATED_DIR = "../curated"

df_movies_full.write.mode("overwrite").parquet(f"{CURATED_DIR}/movies")
df_ratings_clean.write.mode("overwrite").parquet(f"{CURATED_DIR}/ratings")
df_tags_clean.write.mode("overwrite").parquet(f"{CURATED_DIR}/tags")

sorted(os.listdir(CURATED_DIR))

24. Read the three datasets back from disk and check that the row counts and the schemas survived the round trip.

Notice that you did not have to pass any schema this time.

In [ ]:
for dataset in ["movies", "ratings", "tags"]:
    df_curated = spark.read.parquet(f"{CURATED_DIR}/{dataset}")
    print(f"--- {dataset}: {df_curated.count()} rows")
    df_curated.printSchema()

---

## Wrap up

Before moving to Week 2, make sure that:

- [ ] the `../curated/` folder contains the `movies`, `ratings` and `tags` datasets
- [ ] `movies` has 9742 rows, with a clean `title`, an integer `release_year` and the external identifiers
- [ ] `ratings` has 100836 rows, with a `rating_date` instead of a unix timestamp
- [ ] `tags` has 3683 rows, with lowercased and trimmed tags
- [ ] you can describe your findings for the questions in section 7, in one sentence each

In **Week 2** you will pick this curated dataset up and go deeper: window functions, complex data types and UDFs, ending with three open questions where you explain your own findings.